In [1]:
import spacy
import pandas as pd
import numpy as np
import re
import utils
import random
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorForLanguageModeling, AutoModelForMaskedLM, TrainingArguments, Trainer


In [2]:
df = pd.read_csv('gstt_pet_data_2012-22_anon.csv')
print(df.shape)

df_to_drop = pd.read_csv('tnm_all_uncertainty.csv')
acc_to_drop = df_to_drop['metadata.Accession'].to_list()

df = df[~df['Accession'].isin(acc_to_drop)].reset_index()

print(df.shape)

(59729, 13)
(57240, 14)


In [3]:
df2 = df[df['Report text'].str.contains(r"(?s)^.*?(?=~)")]
print(df2.shape)

(21416, 14)


In [4]:
df['Report text'][21853]

', 16:35, NM Halfbody + H&N FDG PET CT\nREFERRAL INFORMATION:\nMetastatic SCC right neck of occult primary\nQuestion/s: Identify primary site ? Tongue base\nADMINISTERED ACTIVITY: 339 MBq\nDESCRIPTION:\nFDG PET scan was performed from base of neck to upper thighs with low dose non-contrast CT for attenuation correction and image fusion. Dedicated views of the head and neck were also obtained.\nThe known right level 2 neck node measuring 1.4 cm in axial diameter demonstrates intense tracer uptake with SUVmax 16.6.\nAlthough there is no definite focal high grade tracer uptake elsewhere within the head and neck there appears to be subtle asymmetrical tracer uptake within the floor of mouth on the right side on PACS fused axial image 44 with no definite CT correlate (best seen on MIP).\nThere is no definite focal tracer uptake within the base of tongue on the right side and also the subtle asymmetrical soft tissue in the right vallecula along the anterior epiglottis demonstrates only faint

In [5]:
ds_sample = pd.DataFrame(df['Report text'])

ds = Dataset.from_pandas(ds_sample)
ds[51]

{'Report text': ', 16:37, NM Halfbody FDG PET CT\n   \nREFERRAL INFORMATION: Metastatic merkel cell. Also multiple SqCC skin. ?progression of liver metastasis.\n\nADMINISTERED ACTIVITY: 296.20 MBq\nBLOOD GLUCOSE: 4.4 mmol/L\n\nDESCRIPTION:\nA half body FDG scan was performed from skull vertex to upper thighs with low dose non-contrast CT for attenuation correction and image fusion.\n\nComparison is made with the previous study of .\n\nThe previously visualised metastasis in the periphery of the right lobe of the liver (junction of segments 7 and 8) has increased slightly in conspicuity and intensity of tracer uptake, SUVmax = 4.8, PACS fused axial image 257. A subtle focus of increased tracer uptake is now also seen inferiorly in the periphery at the junction of segments 4b and 5, PACS fused axial image 282.\n\nA further focus of increased tracer uptake is seen projected over the ascending colon just inferior to the right lobe of the liver, PACS fused axial image 304.\n\nThe previously

In [6]:
model_checkpoint = "UFNLP/gatortron-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

def tokenize_function(examples):
    return tokenizer(examples["Report text"])

tokenized_datasets = ds.map(tokenize_function, batched=True, num_proc=4, remove_columns=["Report text"])

Map (num_proc=4):   0%|          | 0/57240 [00:00<?, ? examples/s]

In [7]:
block_size = 512

def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
        # customize this part to your needs.
    total_length = (total_length // block_size) * block_size
    # Split by chunks of max_len.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4,
)



Map (num_proc=4):   0%|          | 0/57240 [00:00<?, ? examples/s]

In [8]:
lm_datasets = lm_datasets.train_test_split(test_size=0.1)
lm_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 34366
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3819
    })
})

In [9]:
output_dir = "da_gatortron_base_GSTTv2"

model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)
training_args = TrainingArguments(
                                output_dir=output_dir,
                                num_train_epochs=5,
                                save_strategy="epoch",
                                save_total_limit=3,
                                evaluation_strategy="epoch",
                                logging_dir=f"{output_dir}/logs",
                                logging_strategy="epoch",
                                metric_for_best_model='loss',
                                load_best_model_at_end=True,
                                learning_rate=1e-5,
                                per_device_train_batch_size=8,
                                per_device_eval_batch_size=8,
                                weight_decay=0.0,
                                label_names=["labels"],
                                push_to_hub=False,
                                )

In [10]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["test"],
    data_collator=data_collator,
)

In [11]:
trainer.train()

/home/stephen/miniforge3/envs/pytorch/lib/python3.11/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss
1,0.757700,0.648526
2,0.656200,0.618431
3,0.628800,0.600141


TrainOutput(global_step=12888, training_loss=0.680883557954832, metrics={'train_runtime': 8589.4935, 'train_samples_per_second': 12.003, 'train_steps_per_second': 1.5, 'total_flos': 9.609615065795789e+16, 'train_loss': 0.680883557954832, 'epoch': 3.0})

In [12]:
import math
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 1.82


In [13]:
trainer.model.save_pretrained(f"{output_dir}/results")
tokenizer.save_pretrained(f"{output_dir}/results")

('da_gatortron_base_GSTTv2/results/tokenizer_config.json',
 'da_gatortron_base_GSTTv2/results/special_tokens_map.json',
 'da_gatortron_base_GSTTv2/results/vocab.txt',
 'da_gatortron_base_GSTTv2/results/added_tokens.json',
 'da_gatortron_base_GSTTv2/results/tokenizer.json')